### Querying the datasets one by one

In [0]:
dbutils.widgets.text("catalog", "northstar_dev")
dbutils.widgets.text("schema", "raw")
dbutils.widgets.text("volume", "landing")

dbutils.widgets.text("catalog", "northstar_dev")
dbutils.widgets.text("schema", "raw")
dbutils.widgets.text("volume", "landing")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print(f"Catalog.Schema.Volume = {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"Volume root           = {VOLUME_ROOT}")

Catalog.Schema.Volume = northstar_dev.raw.landing
Volume root           = /Volumes/northstar_dev/raw/landing


/Users/as-mac-1375/Documents/GitHub/northstar_pay_lakehouse/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [0]:
CUSTOMERS_PATH = f"{VOLUME_ROOT}/customers"
MERCHANTS_PATH = f"{VOLUME_ROOT}/merchants"
TRANSACTIONS_PATH = f"{VOLUME_ROOT}/transactions"
DEVICE_SESSION_LOGS_PATH = f"{VOLUME_ROOT}/device_session_logs"

print(f"Customers path        = {CUSTOMERS_PATH}")
print(f"Merchants path        = {MERCHANTS_PATH}")
print(f"Transactions path     = {TRANSACTIONS_PATH}")
print(f"Device session logs   = {DEVICE_SESSION_LOGS_PATH}")

Customers path        = /Volumes/northstar_dev/raw/landing/customers
Merchants path        = /Volumes/northstar_dev/raw/landing/merchants
Transactions path     = /Volumes/northstar_dev/raw/landing/transactions
Device session logs   = /Volumes/northstar_dev/raw/landing/device_session_logs


In [0]:
datasets_paths_csv = [CUSTOMERS_PATH, MERCHANTS_PATH, TRANSACTIONS_PATH]
datasets_paths_json = [DEVICE_SESSION_LOGS_PATH]

for path in datasets_paths_csv:
    df = spark.read.format("csv") \
            .option("header", True) \
            .option("recursiveFileLookup", "true") \
            .load(path)

    print(f"Dataset: {path}")
    print(f"Number of records: {df.count()}")
    print(f"Schema:")
    df.printSchema()

    print(f"Sample records:")
#     df.limit(20).display()

for path in datasets_paths_json:
    df = spark.read.format("json") \
            .option("recursiveFileLookup", "true") \
            .load(path)

    print(f"Dataset: {path}")
    print(f"Number of records: {df.count()}")
    print(f"Schema: ")
    df.printSchema()

    print(f"Sample records:")
#     df.limit(20).display()

### Step 1: Data Quality Checks

Runs the following checks across all four datasets (`customers`, `merchants`, `transactions`, `device_session_logs`):

- **Row count** — total number of records in each dataset
- **Full-row duplicates** — rows where every column value is identical; detected via `dropDuplicates()` across all columns
- **PK duplicates** — rows sharing the same primary key but differing in one or more other columns (e.g. same `customer_id`, different `address`); more dangerous than full-row duplicates as they indicate conflicting records for the same entity
- **Null counts per column** — number of nulls in each column displayed as an interactive table; pay particular attention to columns marked `nullfalse` (`customer_id`, `merchant_id`, `transaction_id`, `session_id`, `card_id`, `txn_timestamp`, `event_timestamp`, `last_updated_ts`)

In [0]:
from pyspark.sql import functions as F

# (path, format, pk_cols) — one entry per dataset
DATASETS = [
    (CUSTOMERS_PATH,           "csv",  ["customer_id"]),
    (MERCHANTS_PATH,           "csv",  ["merchant_id"]),
    (TRANSACTIONS_PATH,        "csv",  ["transaction_id"]),
    (DEVICE_SESSION_LOGS_PATH, "json", ["session_id"]),
]

def data_quality_check(df, path, all_cols, pk_cols=None):
    """Run row count, full-row and PK duplicate checks, and null counts.

    all_cols is resolved once per DataFrame in the calling scope and passed in
    to avoid triggering a schema RPC inside the function body.
    """
    name = path.rstrip("/").split("/")[-1].upper()
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")

    # --- Row count ---
    row_count = df.count()
    print(f"  Row count                    : {row_count:,}")

    # --- Full-row duplicates (all columns must match) ---
    full_dup_count = row_count - df.dropDuplicates().count()
    print(f"  Full-row duplicates          : {full_dup_count:,}")

    # --- PK-level duplicates (same PK, any column may differ) ---
    if pk_cols:
        pk_dup_count = row_count - df.dropDuplicates(pk_cols).count()
        label = ", ".join(pk_cols)
        print(f"  PK duplicates ({label:}) : {pk_dup_count:,}")

    # --- Null counts per column ---
    print(f"\n  Null counts per column:")
    null_expr = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in all_cols]
    display(df.select(null_expr))

    # --- Sample records ---
    print(f"\n  Sample records (20 rows):")
    # display(df, limit=20)

for path, fmt, pk_cols in DATASETS:
    reader = spark.read.format(fmt).option("recursiveFileLookup", "true")
    if fmt == "csv":
        reader = reader.option("header", True)
    df = reader.load(path)
    all_cols = df.columns  # schema resolved once per DataFrame, outside the function
    data_quality_check(df, path, all_cols, pk_cols=pk_cols)

### Step 2: Schema Validation
Attempt to cast each string column to its expected type. A **cast failure** is a row where the original value is non-null but the cast returns null — these become silent nulls in the pipeline.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, BooleanType

# --- Cast helpers (col_name -> Column) ---
def _date(c):   return F.to_date(F.col(c), "yyyy-MM-dd")
def _ts(c):     return F.to_timestamp(F.col(c))
def _double(c): return F.col(c).cast(DoubleType())
def _int(c):    return F.col(c).cast(IntegerType())
def _bool(c):   return F.col(c).cast(BooleanType())

# (col_name, cast_fn, expected_type_label) per dataset path
SCHEMA_CHECKS = {
    CUSTOMERS_PATH: [
        ("dob",             _date,   "date"),
        ("signup_date",     _date,   "date"),
        ("last_updated_ts", _ts,     "timestamp"),
    ],
    MERCHANTS_PATH: [
        ("mcc_code",            _int,    "integer"),
        ("onboarding_date",     _date,   "date"),
        ("merchant_risk_score", _double, "double"),
    ],
    TRANSACTIONS_PATH: [
        ("txn_timestamp", _ts,     "timestamp"),
        ("amount",        _double, "double"),
        ("mcc_code",      _int,    "integer"),
        ("is_fraud_flag", _bool,   "boolean"),
        ("txn_date",      _date,   "date"),
    ],
    DEVICE_SESSION_LOGS_PATH: [
        ("event_timestamp", _ts, "timestamp"),
    ],
}

def schema_validation(df, path, checks):
    """Single-pass cast validation.

    For each (col, cast_fn, label), counts rows where the original value
    is non-null but the cast returns null — unparseable values that will
    silently become nulls when the pipeline casts the column.
    """
    name = path.rstrip("/").split("/")[-1].upper()
    print(f"\n{'='*60}")
    print(f"Schema Validation: {name}")
    print(f"{'='*60}")

    # Build all aggregations in a single pass over the DataFrame
    exprs = []
    for col_name, cast_fn, _ in checks:
        cast_col = cast_fn(col_name)
        exprs += [
            F.count(F.when(F.col(col_name).isNotNull(), 1)).alias(f"{col_name}__nn"),
            F.count(F.when(F.col(col_name).isNotNull() & cast_col.isNull(), 1)).alias(f"{col_name}__fail"),
        ]

    stats = df.select(exprs).collect()[0]

    rows = []
    for col_name, _, label in checks:
        non_null = stats[f"{col_name}__nn"]
        fail     = stats[f"{col_name}__fail"]
        pct      = round(fail / non_null * 100, 2) if non_null > 0 else 0.0
        status   = "PASS" if fail == 0 else "FAIL"
        rows.append((col_name, label, non_null, fail, pct, status))

    result_df = spark.createDataFrame(
        rows,
        ["column", "expected_type", "non_null_rows", "cast_failures", "failure_%", "status"]
    )
    display(result_df)

# Run for each dataset — reuses the DATASETS config from Cell 6
for path, fmt, _ in DATASETS:
    reader = spark.read.format(fmt).option("recursiveFileLookup", "true")
    if fmt == "csv":
        reader = reader.option("header", True)
    df = reader.load(path)
    if path in SCHEMA_CHECKS:
        schema_validation(df, path, SCHEMA_CHECKS[path])

### Step 3: Cardinality Checks

Counts distinct values on key columns across all four datasets to answer two questions:

- **Are the master tables clean?** — distinct `customer_id` in `customers` should be ~50,000; distinct `merchant_id` in `merchants` should be ~9,000. A lower number means duplicate PKs survived Step 1's PK duplicate check with different column values
- **What is the fan-out ratio?** — `card_id` and `customer_id` in `transactions` reveal avg transactions per card and per customer; `session_id` and `device_id` in `device_session_logs` reveal avg sessions per device and per customer. These ratios are useful baselines for fraud modelling
- **Cross-table coverage** — compares distinct `customer_id` and `merchant_id` found in `transactions` and `device_session_logs` against their master table counts. A higher count in the child table than the master table signals orphan keys and directly feeds into Step 4 (Referential Integrity)

In [0]:
# ── Step 3: Cardinality Checks ─────────────────────────────────────────────

# Load all four DataFrames once - reused across cardinality checks
dfs = {}
for path, fmt, _ in DATASETS:
    reader = spark.read.format(fmt).option("recursiveFileLookup", "true")
    if fmt == "csv":
        reader = reader.option("header", True)
    dfs[path] = reader.load(path)

df_customers = dfs[CUSTOMERS_PATH]
df_merchants = dfs[MERCHANTS_PATH]
df_transactions = dfs[TRANSACTIONS_PATH]
df_devices = dfs[DEVICE_SESSION_LOGS_PATH]

print("=" * 60)
print("Cardinality Checks")
print("=" * 60)

# --- Master tables ---
cust_distinct = df_customers.select("customer_id").distinct().count()
merch_distinct = df_merchants.select("merchant_id").distinct().count()

print(f"\nCustomers   — distinct customer_id  : {cust_distinct:,}  (expected ~50,000)")
print(f"Merchants   — distinct merchant_id  : {merch_distinct:,}  (expected ~9,000)")

# --- Transactions ---
txn_txn_id = df_transactions.select("transaction_id").distinct().count()
txn_card_id = df_transactions.select("card_id").distinct().count()
txn_cust_id = df_transactions.select("customer_id").distinct().count()
txn_merch_id = df_transactions.select("merchant_id").distinct().count()

txn_rows = df_transactions.count()

print(f"\ntransactions — total rows            : {txn_rows:,}")
print(f"transactions — distinct transaction_id : {txn_txn_id:,}")
print(f"transactions — distinct card_id         : {txn_card_id:,}")
print(f"transactions — distinct customer_id     : {txn_cust_id:,}  (vs {cust_distinct:,} in customers)")
print(f"transactions — distinct merchant_id     : {txn_merch_id:,}  (vs {merch_distinct:,} in merchants)")

avg_txns_per_card = round(txn_rows / txn_card_id, 2) if txn_card_id > 0 else 0
avg_txns_per_cust = round(txn_rows / txn_cust_id, 2) if txn_cust_id > 0 else 0
avg_txns_per_merch = round(txn_rows / txn_merch_id, 2) if txn_merch_id > 0 else 0

print(f"transactions — avg txns per card_id     : {avg_txns_per_card}")
print(f"transactions — avg txns per customer_id : {avg_txns_per_cust}")
print(f"transactions — avg txns per merchant_id : {avg_txns_per_merch}")

# --- Device session logs ---
dev_cust_id = df_devices.select("customer_id").distinct().count()
dev_sess_id = df_devices.select("session_id").distinct().count()
dev_dev_id = df_devices.select("device_id").distinct().count()

dev_rows = df_devices.count()

print(f"\ndevice_session_logs — total rows        : {dev_rows:,}")
print(f"device_session_logs — distinct session_id : {dev_sess_id:,}")
print(f"device_session_logs — distinct device_id  : {dev_dev_id:,}")
print(f"device_session_logs — distinct customer_id : {dev_cust_id:,}  (vs {cust_distinct:,} in customers)")

avg_sess_per_dev = round(dev_rows / dev_dev_id, 2) if dev_dev_id > 0 else 0
avg_sess_per_cust = round(dev_rows / dev_cust_id, 2) if dev_cust_id > 0 else 0

print(f"device_session_logs — avg sessions per device_id : {avg_sess_per_dev}")
print(f"device_session_logs — avg sessions per customer_id : {avg_sess_per_cust}")

### Step 4: Referential Integrity

Verifies that every foreign key in the child tables resolves to a real record in the corresponding master table. An **orphan key** is a value that exists in the child table but has no matching row in the master.

Checks performed:
- **`transactions.customer_id` → `customers.customer_id`** — every transaction must belong to a known customer
- **`transactions.merchant_id` → `merchants.merchant_id`** — every transaction must reference a known merchant
- **`transactions.mcc_code` → `merchants.mcc_code`** — MCC codes used in transactions should exist in the merchants master
- **`device_session_logs.customer_id` → `customers.customer_id`** — every device session must belong to a known customer

Expected orphan rate is **~0%** for all checks. If meaningfully above 0%, the most likely causes are:
- Late-arriving dimension data (master tables not yet updated when child records landed)
- Test or synthetic records that bypassed the normal onboarding flow
- Hard deletes upstream (customer/merchant removed from source but transactions retained)

Orphan counts found here become **blockers** in the Step 10 scorecard if above an acceptable threshold.

In [0]:
# ── Step 4: Referential Integrity ────────────────────────────────────────────
# Reuses df_customers, df_merchants, df_transactions, df_devices from Cell 10.
# Null FK values are excluded from orphan counts — those are flagged in Step 1.

RI_CHECKS = [
    # (child_df, child_col, master_df, master_col, label)
    (df_transactions, "customer_id", df_customers, "customer_id", "transactions.customer_id → customers.customer_id"),
    (df_transactions, "merchant_id", df_merchants, "merchant_id", "transactions.merchant_id → merchants.merchant_id"),
    (df_transactions, "mcc_code",    df_merchants, "mcc_code",    "transactions.mcc_code → merchants.mcc_code"),
    (df_devices,      "customer_id", df_customers, "customer_id", "device_session_logs.customer_id → customers.customer_id"),
]

print("=" * 60)
print("Referential Integrity Checks")
print("=" * 60)

results = []
for child_df, child_col, master_df, master_col, label in RI_CHECKS:
    total = child_df.filter(F.col(child_col).isNotNull()).count()

    # Alias master key to avoid column ambiguity when names are identical
    master_keys = master_df.select(F.col(master_col).alias("__key")).distinct()
    orphan_count = (
        child_df
        .filter(F.col(child_col).isNotNull())
        .join(master_keys, F.col(child_col) == F.col("__key"), "left_anti")
        .count()
    )

    orphan_pct = round(orphan_count / total * 100, 4) if total > 0 else 0.0
    status     = "PASS" if orphan_count == 0 else "FAIL"
    results.append((label, total, orphan_count, orphan_pct, status))

    print(f"\n  {label}")
    print(f"    Non-null rows  : {total:,}")
    print(f"    Orphan rows    : {orphan_count:,}")
    print(f"    Orphan rate    : {orphan_pct}%")
    print(f"    Status         : {status}")

# Summary table
print(f"\n{'='*60}")
print("Summary")
print(f"{'='*60}")
summary_df = spark.createDataFrame(
    results,
    ["check", "non_null_rows", "orphan_rows", "orphan_%", "status"]
)
display(summary_df)

### Step 5: Distribution Analysis

Examines the shape and spread of key columns across `transactions` and `merchants`
to establish baselines and surface anomalies before pipeline development.

- **Amount distribution** — min, max, mean, median, and std dev of `amount`; flags
  negative values (possible refunds — valid or bad data?) and zero-amount transactions
- **MCC code frequency** — top-10 value counts on `mcc_code` in both `transactions`
  and `merchants` to confirm they use the same coding scheme and identify dominant categories
- **Channel breakdown** — value counts on `channel` (e.g. online/POS/ATM) with %
  share; reveals the transaction type mix
- **Status breakdown** — value counts on `status` (approved/declined/pending) with %
  share; establishes base approval rates
- **Fraud rate** — precise class balance on `is_fraud_flag`; this is the primary ML
  target label so the imbalance ratio must be noted prominently in the scorecard

In [0]:
# ── Step 5: Distribution Analysis ─────────────────────────────────────────────
# Reuses df_transactions, df_merchants from Cell 10.

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

print("=" * 60)
print("Distribution Analysis")
print("=" * 60)

# ── 1. Amount stats ────────────────────────────────────────────────────────────
# Cast inline — does not modify df_transactions
amount_col = F.col("amount").cast(DoubleType())

print("\n--- Transaction Amount ---")
amount_stats = df_transactions.select(
    F.count(F.when(amount_col.isNotNull(), 1)).alias("non_null_count"),
    F.min(amount_col).alias("min"),
    F.max(amount_col).alias("max"),
    F.round(F.mean(amount_col), 4).alias("mean"),
    F.round(F.stddev(amount_col), 4).alias("stddev"),
    F.count(F.when(amount_col < 0, 1)).alias("negative_count"),
    F.count(F.when(amount_col == 0, 1)).alias("zero_count")
).collect()[0]

print(f"  Non-null rows   : {amount_stats['non_null_count']:,}")
print(f"  Min             : {amount_stats['min']}")
print(f"  Max             : {amount_stats['max']}")
print(f"  Mean            : {amount_stats['mean']}")
print(f"  Std dev         : {amount_stats['stddev']}")
print(f"  Negative values : {amount_stats['negative_count']:,}  (possible refunds — verify intent)")
print(f"  Zero values     : {amount_stats['zero_count']:,}  (flag if unexpected)")

# percentile_approx must be a separate action — cannot be mixed with above aggregations
median_val = df_transactions.select(
    F.percentile_approx(amount_col, 0.5).alias("median")
).collect()[0]["median"]

print(f"  Median          : {median_val}")

# ── 2. MCC code frequency ──────────────────────────────────────────────────────
print("\n--- MCC Code Frequency: transactions (top 10) ---")
display(
    df_transactions.groupBy("mcc_code")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
)

print("\n--- MCC Code Frequency: merchants (top 10) ---")
display(
    df_merchants.groupBy("mcc_code")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
)


# ── 3. Channel breakdown ───────────────────────────────────────────────────────
print("\n--- Channel Breakdown ---")
txn_total = df_transactions.count()

display(
    df_transactions.groupBy("channel")
        .count()
        .withColumn("pct%", F.round(F.col("count") / txn_total * 100, 2))
        .orderBy(F.desc("count"))
)

# ── 4. Status breakdown ────────────────────────────────────────────────────────
print("\n--- Transaction Status Breakdown ---")
display(
    df_transactions.groupBy("status")
        .count()
        .withColumn("pct%", F.round(F.col("count") / txn_total * 100, 2))
        .orderBy(F.desc("count"))
)

# ── 5. Fraud rate ──────────────────────────────────────────────────────────────
print("\n--- Fraud Rate (is_fraud_flag) ---")
display(
    df_transactions.groupBy("is_fraud_flag")
    .count()
    .withColumn("pct_%", F.round(F.col("count") / txn_total * 100, 4))
    .orderBy("is_fraud_flag")
)

### Step 6: Time Validation

Validates the temporal integrity of date and timestamp columns across all datasets.
A time anomaly at this stage almost always indicates either a formatting inconsistency
in the raw data or an upstream pipeline bug.

- **Range sanity check** — min/max for every date and timestamp column; flag anything
  in 1970, 2099, or clearly outside the expected operational window
- **Future-dated transactions** — any `txn_timestamp` or `txn_date` beyond today is
  invalid and must be flagged as a blocker
- **Pre-signup transactions** — join `transactions` to `customers` and flag any row
  where `txn_date < signup_date`; a classic integrity bug in raw landing data
- **Daily volume gaps** — bucket transactions by day and build a full date spine between
  min/max date; any day with zero transactions is a potential ingestion outage
- **DOB sanity** — flag customers whose `dob` implies age < 18 or age > 100; both
  suggest dirty or synthetic data

In [0]:
# ── Step 6: Time Validation ────────────────────────────────────────────────────
# Reuses df_customers, df_merchants, df_transactions, df_devices from Cell 10.
# Cast helpers (_ts, _date) reused from Cell 8 (Schema Validation).

from datetime import date
TODAY = F.lit(str(date.today()))

print("=" * 60)
print("Time Validation")
print("=" * 60)

# ── 1. Date / timestamp range checks ──────────────────────────────────────────
print("\n--- Date / Timestamp Ranges ---")

RANGE_CHECKS = [
    (df_transactions, "txn_timestamp",   _ts,   "transactions"),
    (df_transactions, "txn_date",        _date, "transactions"),
    (df_customers,    "signup_date",     _date, "customers"),
    (df_customers,    "dob",             _date, "customers"),
    (df_customers,    "last_updated_ts", _ts,   "customers"),
    (df_merchants,    "onboarding_date", _date, "merchants"),
    (df_devices,      "event_timestamp", _ts,   "device_session_logs"),
]

range_rows = []
for df, col_name, cast_fn, table_name in RANGE_CHECKS:
    cast_col = cast_fn(col_name)
    row = df.select(
        F.min(cast_col).alias("min"),
        F.max(cast_col).alias("max"),
    ).collect()[0]
    range_rows.append((table_name, col_name, str(row["min"]), str(row["max"])))

display(spark.createDataFrame(range_rows, ["table", "column", "min", "max"]))

# ── 2. Future-dated transactions ───────────────────────────────────────────────
print("\n--- Future-Dated Transactions ---")

future_count = df_transactions.filter(
    _ts("txn_timestamp").cast("date") > TODAY
).count()

print(f"  txn_timestamp > today : {future_count:,}  ({'FAIL' if future_count > 0 else 'PASS'})")

# ── 3. Pre-signup transactions ─────────────────────────────────────────────────
print("\n--- Transactions Before Customer Signup Date ---")

pre_signup_count = (
    df_transactions
    .select("transaction_id", "customer_id",
        _date("txn_date").alias("txn_d")
    )
    .join(
        df_customers.select("customer_id",
            _date("signup_date").alias("signup_d")
        ),
        on="customer_id",
        how="inner"
    )
    .filter(F.col("txn_d") < F.col("signup_d"))
    .count()
)

print(f"  txn_date < signup_date : {pre_signup_count:,}  ({'FAIL' if pre_signup_count > 0 else 'PASS'})")
      
# ── 4. Daily transaction volume — gap detection ────────────────────────────────
print("\n--- Daily Transaction Volume ---")

daily_vol = (
    df_transactions
    .withColumn("txn_day", _ts("txn_timestamp").cast("date"))
    .groupBy("txn_day")
    .count()
    .orderBy("txn_day")
)

display(daily_vol)

# Build date spine and find days with zero transactions
min_max = daily_vol.select(
    F.min("txn_day").alias("mn"),   F.max("txn_day").alias("mx")
).collect()[0]

date_spine = spark.sql(f"""
    SELECT explode(sequence(DATE '{min_max["mn"]}', DATE '{min_max["mx"]}', INTERVAL 1 DAY))
    AS txn_day
""")

gap_days = date_spine.join(daily_vol, on="txn_day", how="left_anti")
gap_count = gap_days.count()

print(f"\n  Days with zero transactions : {gap_count:,}  ({'FAIL — possible ingestion gap' if gap_count > 0 else 'PASS'})")
if gap_count > 0:
    print("  Gap dates:")
    display(gap_days.orderBy("txn_day"))

# ── 5. DOB sanity (customers) ──────────────────────────────────────────────────
print("\n--- DOB Sanity Check ---")

dob_issues = df_customers.select(
    F.count(F.when(_date("dob") > TODAY, 1)).alias("future_dob"),
    F.count(F.when(F.months_between(TODAY, _date("dob")) / 12 < 18, 1)).alias("under_18"),
    F.count(F.when(F.months_between(TODAY, _date("dob")) / 12 > 100, 1)).alias("over_100"),
).collect()[0]

print(f"  Future DOB (dob > today) : {dob_issues['future_dob']:,}  ({'FAIL' if dob_issues['future_dob'] > 0 else 'PASS'})")
print(f"  Age < 18                 : {dob_issues['under_18']:,}  ({'FLAG' if dob_issues['under_18'] > 0 else 'PASS'})")
print(f"  Age > 100                : {dob_issues['over_100']:,}  ({'FLAG' if dob_issues['over_100'] > 0 else 'PASS'})")

### Step 7: Semi-Structured Validation (device_session_logs)

Validates the nested `device_meta` and `geo` structs in `device_session_logs`, which
cannot be covered by the flat-column checks in earlier steps.

- **Flattened sample** — explode both structs into top-level columns and visually
  inspect a 20-row sample; confirms sub-fields are populated and sensible at a glance
- **Nested null checks** — two levels of nulls to distinguish: (a) entire struct is
  null vs (b) struct is present but a key sub-field like `geo.lat` or `geo.long` is null.
  These are different failure modes with different root causes
- **Geo coordinate range validation** — `geo.lat` must be within -90 to 90;
  `geo.long` must be within -180 to 180; anything outside is physically impossible
  and likely a parsing or encoding bug
- **Schema drift** — compares null rates per sub-field between the oldest 100 and
  newest 100 records (by `event_timestamp`); a field that was always populated in old
  records but goes null in new ones indicates a field was renamed or dropped upstream

In [0]:
# ── Step 7: Semi-Structured Validation ────────────────────────────────────────
# Reuses df_devices from Cell 10.
# Cast helper _ts reused from Cell 8.

print("=" * 60)
print("Semi-Structured Validation: device_session_logs")
print("=" * 60)

# ── 1. Flattened sample ────────────────────────────────────────────────────────
print("\n--- Flattened Sample (20 rows) ---")

df_flat = df_devices.select(
    "session_id",
    "customer_id",
    "device_id",
    "event_type",
    "event_timestamp",
    "ip_address",
    F.col("device_meta.app_version").alias("app_version"),
    F.col("device_meta.device_model").alias("device_model"),
    F.col("device_meta.os").alias("os"),
    F.col("geo.city").alias("city"),
    F.col("geo.lat").alias("lat"),
    F.col("geo.long").alias("long"),
    F.col("geo.accuracy_m").alias("accuracy_m"),
)

df_flat.limit(20).display()

# ── 2. Nested null checks ──────────────────────────────────────────────────────
# Level 1: entire struct null
# Level 2: struct present but sub-field null
print("\n--- Nested Null Checks ---")

nested_nulls = df_devices.select(
    F.count(F.when(F.col("device_meta").isNull(), 1).alias("device_meta__struct_null")),
    F.count(F.when(F.col("geo").isNull(), 1).alias("geo__struct_null")),

    F.count(F.when(F.col("device_meta").isNotNull() & F.col("device_meta.app_version").isNull(), 1).alias("device_meta__app_version_null")),
    F.count(F.when(F.col("device_meta").isNotNull() & F.col("device_meta.device_model").isNull(), 1).alias("device_meta__device_model_null")),
    F.count(F.when(F.col("device_meta").isNotNull() & F.col("device_meta.os").isNull(), 1).alias("device_meta__os_null")),

    F.count(F.when(F.col("geo").isNotNull() & F.col("geo.lat").isNull(), 1).alias("geo__lat_null")),
    F.count(F.when(F.col("geo").isNotNull() & F.col("geo.long").isNull(), 1).alias("geo__long_null")),
    F.count(F.when(F.col("geo").isNotNull() & F.col("geo.city").isNull(), 1).alias("geo__city_null")),
    F.count(F.when(F.col("geo").isNotNull() & F.col("geo.accuracy_m").isNull(), 1).alias("geo__accuracy_m_null")),
).collect()[0]

for field, val in nested_nulls.asDict().items():
    status = "FAIL" if val > 0 else "PASS"
    print(f"  {field:<45} : {val:,}  ({status})")

# ── 3. Geo coordinate range validation ────────────────────────────────────────
print("\n--- Geo Coordinate Range Validation ---")

geo_stats = df_devices.filter(F.col("geo").isNotNull()).select(
    F.count(F.when((F.col("geo.lat") < -90) | (F.col("geo.lat") > 90), 1)).alias("invalid_lat"),
    F.count(F.when((F.col("geo.long") < -180) | (F.col("geo.long") > 180), 1)).alias("invalid_long"),
    
    F.min("geo.lat").alias("lat_min"),
    F.max("geo.lat").alias("lat_max"),
    F.min("geo.long").alias("long_min"),
    F.max("geo.long").alias("long_max"),
).collect()[0]

print(f"  Invalid lat  (-90 to 90)    : {geo_stats['invalid_lat']:,}  ({'FAIL' if geo_stats['invalid_lat'] > 0 else 'PASS'})")
print(f"  Invalid long (-180 to 180)  : {geo_stats['invalid_long']:,}  ({'FAIL' if geo_stats['invalid_long'] > 0 else 'PASS'})")
print(f"  Lat range    : {geo_stats['lat_min']} → {geo_stats['lat_max']}")
print(f"  Long range   : {geo_stats['long_min']} → {geo_stats['long_max']}")

# ── 4. Schema drift: oldest 100 vs newest 100 ─────────────────────────────────
# Null rate difference per sub-field is a proxy for upstream field rename/removal
print("\n--- Schema Drift: oldest 100 vs newest 100 rows ---")

ts_col = _ts("event_timestamp")
SUB_FIELDS = ["app_version", "device_model", "os", "lat", "long", "city", "accuracy_m"]

oldest_100 = df_flat.orderBy(ts_col.asc()).limit(100)
newest_100 = df_flat.orderBy(ts_col.desc()).limit(100)

drift_rows = []
for field in SUB_FIELDS:
    old_nulls = oldest_100.filter(F.col(field).isNull()).count()
    new_nulls = newest_100.filter(F.col(field).isNull()).count()

    status = "DRIFT" if old_nulls != new_nulls else "OK"
    drift_rows.append((field, old_nulls, new_nulls, status))

display(spark.createDataFrame(drift_rows, ["sub_field", "null_oldest_100", "null_newest_100", "status"]))

### Step 8: Outlier Detection

Identifies extreme or implausible values that pass type validation but represent real data quality issues.

- **Transaction amount outliers** — flags all transactions above the 99.9th percentile by amount; these may be legitimate high-value transactions or data entry errors and should be eyeballed before the pipeline is built
- **Impossible geographic velocity** — for each customer, sorts device sessions by `event_timestamp`, computes Haversine distance and implied travel speed between consecutive sessions; any pair implying speed > 900 km/h (commercial aviation max) is physically impossible and likely indicates a compromised account, a bot, or synthetic data

In [0]:
# ── Step 8: Outlier Detection ──────────────────────────────────────────────
# Reuses df_transactions, df_devices from Cell 10.
# Cast helpers (_ts, _date, _double) reused from Cell 8.

from pyspark.sql import Window

print("=" * 60)
print("Outlier Detection")
print("=" * 60)

# ── 1. Transaction amount outliers (top 0.1%) ──────────────────────────
print("\n--- Transaction Amount Outliers (top 0.1%) ---")

amount_col = _double("amount")

p999 = df_transactions.select(
    F.percentile_approx(amount_col, 0.999).alias("p999")
).collect()[0]["p999"]

print(f"  99.9th percentile threshold : {p999}")

outlier_txns = (
    df_transactions
    .withColumn("amount_cast", amount_col)
    .filter(F.col("amount_cast") > p999)
    .select("transaction_id", "customer_id", "merchant_id",
            "amount_cast", "txn_timestamp", "status", "is_fraud_flag")
    .orderBy(F.desc("amount_cast"))
)

outlier_count = outlier_txns.count()
print(f"  Outlier transactions        : {outlier_count:,}")
outlier_txns.limit(20).display()

# ── 2. Impossible geographic velocity ─────────────────────────────────
# Haversine distance between consecutive sessions per customer.
# Flag pairs where implied speed > 900 km/h (commercial aviation max).
print("\n--- Impossible Geographic Velocity (> 900 km/h) ---")

MAX_SPEED_KMH = 900
EARTH_RADIUS_KM = 6371

# Only include rows with valid geo coordinates
df_geo = (
    df_devices
    .filter(
        F.col("geo").isNotNull() &
        F.col("geo.lat").isNotNull() &
        F.col("geo.long").isNotNull()
    )
    .select(
        "customer_id",
        "session_id",
        _ts("event_timestamp").alias("event_ts"),
        F.col("geo.lat").alias("lat"),
        F.col("geo.long").alias("lng"),
    )
)

# Lag window: previous session per customer ordered by timestamp
w = Window.partitionBy("customer_id").orderBy("event_ts")

df_pairs = (
    df_geo
    .withColumn("prev_lat", F.lag("lat",  1).over(w))
    .withColumn("prev_lng", F.lag("lng",  1).over(w))
    .withColumn("prev_ts",  F.lag("event_ts", 1).over(w))
    .filter(F.col("prev_lat").isNotNull())
)

# Haversine formula
df_velocity = df_pairs.withColumn(
    "dist_km",
    2 * EARTH_RADIUS_KM * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("lat") - F.col("prev_lat")) / 2), 2) +
        F.cos(F.radians(F.col("prev_lat"))) *
        F.cos(F.radians(F.col("lat"))) *
        F.pow(F.sin(F.radians(F.col("lng") - F.col("prev_lng")) / 2), 2)
    ))
).withColumn(
    "time_hours",
    # Floor at 1 second to avoid division by zero on same-second events
    F.greatest(
        (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / 3600.0,
        F.lit(1 / 3600.0)
    )
).withColumn(
    "speed_kmh",
    F.round(F.col("dist_km") / F.col("time_hours"), 2)
)

impossible = df_velocity.filter(F.col("speed_kmh") > MAX_SPEED_KMH)
impossible_count = impossible.count()

print(f"  Impossible velocity pairs : {impossible_count:,}  ({'FAIL' if impossible_count > 0 else 'PASS'})")
if impossible_count > 0:
    display(
        impossible
        .select("customer_id", "session_id", "event_ts",
                "lat", "lng", "prev_ts", "prev_lat", "prev_lng",
                "dist_km", "time_hours", "speed_kmh")
        .orderBy(F.desc("speed_kmh"))
        .limit(20)
    )

### Step 9: PII Identification

Documents all columns containing personally identifiable information (PII) across the four datasets. This inventory becomes the direct input for masking, access-control, and data governance decisions before the data moves further downstream.

Sensitivity levels used:
- **High** — direct identifiers: full name, date of birth, precise location (lat/long), IP address. Exposure of these fields directly identifies an individual
- **Medium** — indirect identifiers: values that identify a person only via a join or pattern (customer_id, card_id, device_id). Still regulated under GDPR as pseudonymous data
- **Low** — business PII: identifies a business entity rather than a natural person (merchant_name)

In [0]:
# ── Step 9: PII Identification ──────────────────────────────────────────────

print("=" * 60)
print("PII Inventory")
print("=" * 60)

# (table, column, sensitivity, notes)
PII_INVENTORY = [
    ("customers",           "full_name",   "High",   "Direct identifier — personal name"),
    ("customers",           "dob",         "High",   "Direct identifier — date of birth"),
    ("customers",           "address",     "High",   "Direct identifier — physical address"),
    ("customers",           "customer_id", "Medium", "Indirect identifier — links to all other tables"),
    ("merchants",           "merchant_name","Low",   "Business PII — identifies a business, not an individual"),
    ("transactions",        "customer_id", "Medium", "Indirect identifier — FK to customers"),
    ("transactions",        "card_id",     "Medium", "Indirect identifier — card-level fingerprint linkable to a person"),
    ("device_session_logs", "ip_address",  "High",   "Direct identifier — PII under GDPR"),
    ("device_session_logs", "geo.lat",     "High",   "Precise location — PII under GDPR"),
    ("device_session_logs", "geo.long",    "High",   "Precise location — PII under GDPR"),
    ("device_session_logs", "geo.city",    "Medium", "Coarse location — lower sensitivity than lat/long"),
    ("device_session_logs", "device_id",   "Medium", "Device fingerprint — linkable to a person over time"),
    ("device_session_logs", "customer_id", "Medium", "Indirect identifier — FK to customers"),
]

pii_df = spark.createDataFrame(
    PII_INVENTORY,
    ["table", "column", "sensitivity", "notes"]
)

display(pii_df.orderBy(
    F.when(F.col("sensitivity") == "High", 1)
     .when(F.col("sensitivity") == "Medium", 2)
     .otherwise(3),
    "table", "column"
))

high   = sum(1 for *_, s, _ in PII_INVENTORY if s == "High")
medium = sum(1 for *_, s, _ in PII_INVENTORY if s == "Medium")
low    = sum(1 for *_, s, _ in PII_INVENTORY if s == "Low")

print(f"\n  Total PII columns : {len(PII_INVENTORY)}")
print(f"  High              : {high}")
print(f"  Medium            : {medium}")
print(f"  Low               : {low}")
print("\n  This inventory is the input for masking/access-control decisions before data moves downstream.")

### Step 10: Data Quality Scorecard

Aggregates all findings from Steps 1–9 into a single readiness report. Pulls computed variables directly from earlier cells — no data re-scanned.

- **Section 1** — Dataset overview: row counts and distinct PK counts per table
- **Section 2** — Data quality summary: cardinality ratios from Step 3
- **Section 3** — Referential integrity: orphan rates and pass/fail per FK check from Step 4
- **Section 4** — Key distributions: amount stats from Step 5
- **Section 5** — Temporal validation: future-dated, pre-signup, gap days, and DOB flags from Step 6
- **Section 6** — Outlier analysis: amount outliers and impossible geo velocity from Step 8
- **Section 7** — PII inventory: all tagged columns sorted by sensitivity from Step 9
- **Section 8** — Deployment readiness verdict: RED / YELLOW / GREEN with explicit blockers and warnings

In [0]:
# ── Step 10: Data Quality Scorecard ────────────────────────────────────────────
# Aggregates findings from Steps 1–9. Run all prior steps before executing this cell.

from datetime import datetime

REPORT_TS = datetime.now().strftime("%Y-%m-%d %H:%M")

# ── Safe accessors ────────────────────────────────────────────────────────────
def _v(name, default="N/A"):
    """Get a scalar variable from globals with fallback."""
    return globals().get(name, default)

def _rv(row_var, key, default="N/A"):
    """Get a key from a Row variable stored in globals."""
    row = globals().get(row_var)
    if row is None:
        return default
    try:
        return row[key]
    except Exception:
        return default

def _flag(val, threshold=0, fail="FAIL", pass_="PASS"):
    if not isinstance(val, (int, float)):
        return "N/A"
    return fail if val > threshold else pass_

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  DATA QUALITY SCORECARD  —  NorthstarPay Raw Landing Zone")
print(f"  Generated : {REPORT_TS}")
print(SEP)

# ── SECTION 1: Dataset Overview ───────────────────────────────────────────
print(f"\nSECTION 1 — DATASET OVERVIEW")
print(SEP2)
display(spark.createDataFrame([
    ("customers",           50_000,                    str(_v("cust_distinct"))),
    ("merchants",           9_000,                     str(_v("merch_distinct"))),
    ("transactions",        _v("txn_rows",   525_292), str(_v("txn_txn_id"))),
    ("device_session_logs", _v("dev_rows",   261_414), str(_v("dev_sess_id"))),
], ["table", "total_rows", "distinct_pk"]))

# ── SECTION 2: Data Quality Summary ──────────────────────────────────
print(f"\nSECTION 2 — DATA QUALITY SUMMARY")
print(SEP2)
print("  Per-column null counts and duplicate counts: see Step 1 cell output.")
_txn = _v("txn_rows")
_dev = _v("dev_rows")
if isinstance(_txn, int):
    print(f"\n  transactions:")
    print(f"    Total rows              : {_txn:,}")
    print(f"    Avg txns per customer   : {_v('avg_txns_per_cust')}")
    print(f"    Avg txns per card       : {_v('avg_txns_per_card')}")
if isinstance(_dev, int):
    print(f"\n  device_session_logs:")
    print(f"    Total rows              : {_dev:,}")
    print(f"    Avg sessions per cust   : {_v('avg_sess_per_cust')}")
    print(f"    Avg sessions per device : {_v('avg_sess_per_dev')}")

# ── SECTION 3: Referential Integrity ───────────────────────────────────
print(f"\nSECTION 3 — REFERENTIAL INTEGRITY")
print(SEP2)
ri = _v("results", [])
if ri and isinstance(ri, list):
    display(spark.createDataFrame(ri, ["check", "non_null_rows", "orphan_rows", "orphan_%", "status"]))
else:
    print("  Not available — run Step 4 first.")

# ── SECTION 4: Key Distributions ───────────────────────────────────────
print(f"\nSECTION 4 — KEY DISTRIBUTIONS")
print(SEP2)
print("  Transaction Amount:")
print(f"    Min             : {_rv('amount_stats', 'min')}")
print(f"    Max             : {_rv('amount_stats', 'max')}")
print(f"    Mean            : {_rv('amount_stats', 'mean')}")
print(f"    Median          : {_v('median_val')}")
print(f"    Std dev         : {_rv('amount_stats', 'stddev')}")
print(f"    Negative values : {_rv('amount_stats', 'negative_count')}")
print(f"    Zero values     : {_rv('amount_stats', 'zero_count')}")
print("\n  Channel, status breakdown, and fraud rate: see Step 5 cell output.")

# ── SECTION 5: Temporal Validation ──────────────────────────────────────
print(f"\nSECTION 5 — TEMPORAL VALIDATION FLAGS")
print(SEP2)
_future  = _v("future_count")
_presign = _v("pre_signup_count")
_gaps    = _v("gap_count")
_fut_dob = _rv("dob_issues", "future_dob")
_u18     = _rv("dob_issues", "under_18")
_o100    = _rv("dob_issues", "over_100")
print(f"  Future-dated transactions      : {_future}  ({_flag(_future)})")
print(f"  Pre-signup transactions         : {_presign}  ({_flag(_presign)})")
print(f"  Days with zero transactions     : {_gaps}  ({_flag(_gaps, fail='WARN')})")
print(f"  Future DOB                      : {_fut_dob}  ({_flag(_fut_dob)})")
print(f"  Customers age < 18              : {_u18}  ({_flag(_u18, fail='FLAG')})")
print(f"  Customers age > 100             : {_o100}  ({_flag(_o100, fail='FLAG')})")

# ── SECTION 6: Outlier Analysis ─────────────────────────────────────────────
print(f"\nSECTION 6 — OUTLIER ANALYSIS")
print(SEP2)
_outliers = _v("outlier_count")
_velocity = _v("impossible_count")
print(f"  Amount outliers (top 0.1%)      : {_outliers}  (see Step 8 for full list)")
print(f"  Impossible geo velocity pairs   : {_velocity}  ({_flag(_velocity)})")

# ── SECTION 7: PII Inventory ───────────────────────────────────────────────
print(f"\nSECTION 7 — PII INVENTORY")
print(SEP2)
pii = _v("PII_INVENTORY", [])
if pii:
    display(
        spark.createDataFrame(pii, ["table", "column", "sensitivity", "notes"])
        .orderBy(
            F.when(F.col("sensitivity") == "High",   1)
             .when(F.col("sensitivity") == "Medium", 2)
             .otherwise(3),
            "table", "column"
        )
    )
    print(f"  Total: {len(pii)}  |  High: {_v('high')}  |  Medium: {_v('medium')}  |  Low: {_v('low')}")
else:
    print("  Not available — run Step 9 first.")

# ── SECTION 8: Deployment Readiness Verdict ──────────────────────────────
print(f"\nSECTION 8 — DEPLOYMENT READINESS VERDICT")
print(SEP2)

blockers = []
warnings = []

# Hard blockers
if isinstance(_future,  int) and _future  > 0:
    blockers.append(f"Future-dated transactions              : {_future:,}")
if isinstance(_presign, int) and _presign > 0:
    blockers.append(f"Pre-signup transactions                 : {_presign:,}")
if isinstance(_velocity, int) and _velocity > 0:
    blockers.append(f"Impossible geographic velocity pairs    : {_velocity:,}")
if isinstance(ri, list):
    for row in ri:
        if row[4] == "FAIL":
            blockers.append(f"Referential integrity FAIL — {row[0]} : {row[2]:,} orphans")

# Warnings
if isinstance(_fut_dob, int) and _fut_dob > 0:
    warnings.append(f"Future DOB records                     : {_fut_dob:,}")
if isinstance(_u18, int) and _u18 > 0:
    warnings.append(f"Customers with age < 18                : {_u18:,}")
if isinstance(_o100, int) and _o100 > 0:
    warnings.append(f"Customers with age > 100               : {_o100:,}")
if isinstance(_gaps, int) and _gaps > 0:
    warnings.append(f"Days with zero transactions            : {_gaps:,}  (possible ingestion gap)")
if isinstance(_outliers, int) and _outliers > 0:
    warnings.append(f"Amount outliers (top 0.1%)             : {_outliers:,}  (review before pipeline)")

# Verdict
if blockers:
    verdict = "[RED]    NOT READY — resolve blockers before pipeline development"
elif warnings:
    verdict = "[YELLOW] CONDITIONAL — investigate warnings before production"
else:
    verdict = "[GREEN]  READY FOR PIPELINE DEVELOPMENT"

print(f"\n  VERDICT: {verdict}\n")

if blockers:
    print("  BLOCKERS — must resolve before proceeding:")
    for b in blockers:
        print(f"    [X] {b}")

if warnings:
    print("\n  WARNINGS — investigate before production:")
    for w in warnings:
        print(f"    [!] {w}")

if not blockers and not warnings:
    print("  All checks passed. Data is clean and consistent across all four datasets.")

print(f"\n{SEP}")
print("  END OF SCORECARD")
print(SEP)

  DATA QUALITY SCORECARD  —  NorthstarPay Raw Landing Zone
  Generated : 2026-07-01 13:20

SECTION 1 — DATASET OVERVIEW
----------------------------------------------------------------------


table,total_rows,distinct_pk
customers,50000,50000
merchants,9000,9000
transactions,525292,525292
device_session_logs,261414,261414



SECTION 2 — DATA QUALITY SUMMARY
----------------------------------------------------------------------
  Per-column null counts and duplicate counts: see Step 1 cell output.

  transactions:
    Total rows              : 525,292
    Avg txns per customer   : 11.16
    Avg txns per card       : 6.7

  device_session_logs:
    Total rows              : 261,414
    Avg sessions per cust   : 5.26
    Avg sessions per device : 2.96

SECTION 3 — REFERENTIAL INTEGRITY
----------------------------------------------------------------------


check,non_null_rows,orphan_rows,orphan_%,status
transactions.customer_id → customers.customer_id,524740,0,0.0,PASS
transactions.merchant_id → merchants.merchant_id,525292,0,0.0,PASS
transactions.mcc_code → merchants.mcc_code,525292,0,0.0,PASS
device_session_logs.customer_id → customers.customer_id,261414,0,0.0,PASS



SECTION 4 — KEY DISTRIBUTIONS
----------------------------------------------------------------------
  Transaction Amount:
    Min             : -1590.23
    Max             : 14366.62
    Mean            : 46.9607
    Median          : 25.06
    Std dev         : 128.0401
    Negative values : 526
    Zero values     : 0

  Channel, status breakdown, and fraud rate: see Step 5 cell output.

SECTION 5 — TEMPORAL VALIDATION FLAGS
----------------------------------------------------------------------
  Future-dated transactions      : 0  (PASS)
  Pre-signup transactions         : 107232  (FAIL)
  Days with zero transactions     : 0  (PASS)
  Future DOB                      : 0  (PASS)
  Customers age < 18              : 0  (PASS)
  Customers age > 100             : 0  (PASS)

SECTION 6 — OUTLIER ANALYSIS
----------------------------------------------------------------------
  Amount outliers (top 0.1%)      : 573  (see Step 8 for full list)
  Impossible geo velocity pairs   : 1019  (FAI

table,column,sensitivity,notes
customers,address,High,Direct identifier — physical address
customers,dob,High,Direct identifier — date of birth
customers,full_name,High,Direct identifier — personal name
device_session_logs,geo.lat,High,Precise location — PII under GDPR
device_session_logs,geo.long,High,Precise location — PII under GDPR
device_session_logs,ip_address,High,Direct identifier — PII under GDPR
customers,customer_id,Medium,Indirect identifier — links to all other tables
device_session_logs,customer_id,Medium,Indirect identifier — FK to customers
device_session_logs,device_id,Medium,Device fingerprint — linkable to a person over time
device_session_logs,geo.city,Medium,Coarse location — lower sensitivity than lat/long


  Total: 13  |  High: 6  |  Medium: 6  |  Low: 1

SECTION 8 — DEPLOYMENT READINESS VERDICT
----------------------------------------------------------------------

  VERDICT: [RED]    NOT READY — resolve blockers before pipeline development

  BLOCKERS — must resolve before proceeding:
    [X] Pre-signup transactions                 : 107,232
    [X] Impossible geographic velocity pairs    : 1,019

  WARNINGS — investigate before production:
    [!] Amount outliers (top 0.1%)             : 573  (review before pipeline)

  END OF SCORECARD
